In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df["ndvi"]

In [ ]:
import xee
help(xee.EarthEngineBackendEntrypoint)

In [3]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()


Successfully saved authorization token.


In [2]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
WSDemo_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
Park_Lane_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/park_lane_tahoe")
#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-05-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Park_Lane_Boundaries.geometry(), crs='EPSG:3310', scale=30)

In [ ]:
print(ds)

In [ ]:
# 5. Preview the dataset and the results before doing a full run!
# This allows users to inspect the structure and content of the data to ensure it behaves as expected prior to running a full computation.
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    preview_dataset=True
)

In [3]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

list_of_column_names = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    max_pixels_per_tile = 1_000_000,
    dataset_config={
        'geometry': Park_Lane_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "tune_function": False,
        "chunks": chunks,
        "max_iterations": None,
        "output_column_names": list_of_column_names
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "PL_NDVI_Tiles",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 6,
        "threads_per_worker": 1,
        "memory_limit": "3g",
        #"ee_max_concurrency": 30
    }
)

Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:59639' processes=6 threads=6, memory=16.76 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] AOI tiling enabled. Streaming tiles in batches...
[robustraster] Processing tile 1 of 1
[robustraster] Running user function...
Exported: PL_NDVI_Tiles\x-1153_-1093_y135250_135280_tile_1__time_20180117T183931.tif with bands [np.str_('ndvi')]
Exported: PL_NDVI_Tiles\x-1153_-1093_y135250_135280_tile_1__time_20180407T183853.tif with bands [np.str_('ndvi')]
Exported: PL_NDVI_Tiles\x-1153_-1093_y135250_135280_tile_1__time_20180306T183909.tif with bands [np.str_('ndvi')]
Exported: PL_NDVI_Tiles\x-1153_-1093_y135250_135280_tile_1__time_20180101T183939.tif with bands [np.str_('ndvi')]
Exported: PL_NDVI_Tiles\x-1153_-1093_y135250_135280_tile_1__time_20180423T183844.tif with bands [np.str_('ndvi')]
Exported: PL_NDVI_Tiles\x-1153_-1093_y135250_135280_tile

c:\anaconda3\envs\rr042_rpy2\lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [ ]:
# Docker version
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    #max_pixels_per_tile = 40_000_000,
    dataset_config={
        'geometry': Park_Lane_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "tune_function": False,
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "PL_Tuned_NDVI_Tiles_30m_40MIL",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 4,
        "threads_per_worker": 1,
        "memory_limit": "3g",
        #"ee_max_concurrency": 30
    },
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/rr042"
)